# 06 - Trajectories on the BASE model: same corpus, same plots, same levers

**What this notebook is for.** The base-model arm of notebook 05: identical figures and
controls, but every trajectory comes from `google/gemma-4-31b` (base, no instruction
tuning). The story ids match notebook 05 exactly (same corpus, same shard naming), so any
figure here can be compared panel-for-panel with its 05 counterpart. **It draws no
verdicts**; the Q3.H1.E1 scoring reads were registered 2026-07-22 and their first tranche
ran on the instruct arm 2026-07-23 (notebook 08). The base-vs-instruct comparison is part
of the registered falsify plan and has not yet been scored.

**Index**

1. What was collected (base-arm specifics)
2. One sequential story, every view, scrubbable across layers
3. One simultaneous story, for contrast
4. Explore more stories (20-story dropdown; full-corpus picker)
5. Registered analysis (pending, deliberately empty)

**Data lineage differences from notebook 05 (read this first).**

- **Model**: `google/gemma-4-31b` (base). The stories were still generated by the INSTRUCT
  model (a base model cannot chat-write stories), so for this arm the generator is not the
  probed model. This is unavoidable and documented on the dataset card.
- **Probes**: 195 per shard, not 207: the 171 **base-corpus-lineage** contrasts plus the 24
  random null directions. Probe-version caveat (TREE Q1.H3.E4b): these are the PRE-padding-fix
  base-corpus bundles; E4b measured non-trivial pre/post-fix rotation of corpus contrasts, so
  fine-grained corpus-column reads carry that caveat. There is NO self-generated lineage for base, for the same reason.
  Figures that used selfgen probes in 05 use the corpus lineage of the same emotions here.
- **Everything else matches 05**: layers {6, 15, 24, 33, 42, 51}, centered-cosine
  convention, fp16 shard schema, right padding.
  HF: `abotresol/emotion-combined-trajectories-gemma-4-31b`.


In [ ]:
# this cell loads the trajectory manifest, probe labels, and run config via fetch()
import json

import numpy as np

from emotion_vectors.artifacts import fetch
from emotion_vectors import trajectory_plots as tp
from emotion_vectors.trajectories import transition_windows

manifest = [json.loads(l) for l in fetch("combined_trajectories_base/manifest.jsonl").read_text().splitlines()]
labels = json.loads(fetch("combined_trajectories_base/probe_labels.json").read_text())
config = json.loads(fetch("combined_trajectories_base/run_config.json").read_text())
LAYERS = config["layers"]
MODEL = "gemma-4-31b"
print(f"{len(manifest)} stories | layers {LAYERS} | {len(labels)} probes | model {MODEL}")


In [ ]:
# this cell sets up token-to-text infrastructure (Peyton's design): the tokenizer for
# per-token hover labels, and the raw stories for the colored text panel under each plot
from IPython.display import HTML, display

from emotion_vectors.trajectories import kept_rows, parse_story, story_id

RAW_BY_ID = {story_id(r): r for r in kept_rows(fetch("combined_stories/stories_raw.jsonl"))}

try:
    from transformers import AutoTokenizer

    _tok = AutoTokenizer.from_pretrained(f"google/{MODEL}")

    def decode_tokens(shard):
        return [_tok.decode([int(i)]) for i in shard["token_ids"]]
except Exception as err:  # gated-model auth missing: hovers fall back to t= indices
    print(f"tokenizer unavailable ({type(err).__name__}); hovers will show token indices only")

    def decode_tokens(shard):
        return None


def story_text_panel(story_id_, mode):
    """Colored text-panel HTML: phase spans colored to match the figure colors."""
    parsed = parse_story(RAW_BY_ID[story_id_]["text"], mode)
    return tp.text_panel_html(parsed.clean_text, parsed.phase_char_starts, parsed.phase_emotions)


## 1. What was collected

Descriptive statistics only. The corpus as measured, before any hypothesis touches it.


In [ ]:
# this cell summarizes the corpus: modes, token counts, categories, phase alignment
from collections import Counter

modes = Counter(m["mode"] for m in manifest)
tokens = np.array([m["n_tokens"] for m in manifest])
cats = Counter(m["category"] for m in manifest)
print(f"modes: {dict(modes)}")
print(f"tokens/story: median {np.median(tokens):.0f}, range {tokens.min()}-{tokens.max()}")
print(f"categories: {dict(cats)}")
seq3 = sum(1 for m in manifest if m["mode"] == "SEQUENTIAL" and len(m["phase_token_starts"]) == 3)
print(f"sequential stories with clean 3-phase alignment: {seq3}/{modes['SEQUENTIAL']}")


**The triple categories.** The 173 emotion triples (Peyton Li's Q3 spec,
`scripts/combined_story_gen/emotions_triples_v1.json`) are structured to make different
failure modes of the trajectory readout distinguishable. The names are the spec's own; the
one-line readings below are inferred from the category names and their membership, since
the spec file carries no prose (the letters skip C, which has no entry in v1):

- **A_superposition** (29 triples): related emotions that can plausibly blend in one state,
  such as offended / disturbed / mortified. Tests whether nearby emotions stay separable.
- **B_conflict** (48): valence-opposed combinations, such as scornful / outraged / elated.
  The sharpest test of crossovers, since the emotions cannot be read as one mood.
- **D_timescale** (32): mixes a lasting state (weary, tired, stuck) with momentary
  reactions. Half its triples carry a non-affect control word.
- **E_arousal_mismatch** (32): pairs high-activation and low-activation emotions, such as
  heartbroken / fulfilled / vibrant. Tests the arousal axis of the circumplex.
- **F_valence_spread** (32): triples spanning the valence range, such as hope / desperate /
  dumbstruck.

`has_nonaffect` marks triples containing a word that is not a core emotion (greedy, stuck):
these are control triples, since a probe that tracks them as well as it tracks emotions is
reading something broader than affect.


In [ ]:
# this cell tabulates the triple categories and the non-affect control counts
import pandas as pd

triples = json.loads(open("../scripts/combined_story_gen/emotions_triples_v1.json").read())
rows = []
for cat in sorted({t["category"] for t in triples}):
    members = [t for t in triples if t["category"] == cat]
    rows.append({
        "category": cat,
        "triples": len(members),
        "with_nonaffect_control": sum(1 for t in members if t["has_nonaffect"]),
        "example": " / ".join(members[0]["emotions"]),
    })
pd.DataFrame(rows)


## 2. One sequential story, every view, scrubbable across layers

Story `t000_seq_p2_2f9faf62` (upset to unsettled to cheerful, 236 tokens), chosen as the
first mid-length cross-valence sequential story in the manifest, before any scoring existed.
Corpus-lineage probes, because the triple's emotions are not all in the 12-emotion selfgen
set. Every figure states the model, the story, and the expected emotion order, and most
carry a **layer slider** over the six captured bands (the explore_layers convention). The
paper's layer story predicts token-level jitter early and scene-level phase structure
mid-late, so drag the slider and look for exactly that contrast.


In [ ]:
# this cell loads the exhibit story's shard and builds the layer-scrubbable lines view
STORY = "t000_seq_p2_2f9faf62"
row = next(m for m in manifest if m["story_id"] == STORY)
shard = np.load(fetch(f"combined_trajectories_base/shards/{STORY}.npz"))
emotions, starts = row["phase_emotions"], row["phase_token_starts"]
PHASES = " -> ".join(f"{e} (t={s})" for e, s in zip(emotions, starts))

from emotion_vectors.interactive import (
    trajectory_heatmap_scrubber,
    trajectory_lines_scrubber,
    trajectory_ternary_scrubber,
)

TOKENS = decode_tokens(shard)
cos_by_layer = {
    k: tp.smooth(tp.story_cosines(shard, labels, emotions, i, lineage="corpus"), window=8)
    for i, k in enumerate(LAYERS)
}
trajectory_lines_scrubber(
    LAYERS, cos_by_layer, emotions, starts, default_layer=33, tokens=TOKENS,
    title=f"{MODEL} | {STORY}<br><sub>expected: {PHASES}. Drag the slider across layers</sub>",
).show()


In [ ]:
# this cell renders Peyton's linked plot+text view: hover a point and the token lights
# up in the text (and back); toggle an emotion to shade the words by its cosine; the
# layer slider drives lines, triangle, and text shading together
from emotion_vectors.linked_trajectory import linked_trajectory_html

display(HTML(linked_trajectory_html(
    cos_by_layer, TOKENS, emotions, starts, default_layer=33, show_ternary=True,
    title=f"{MODEL}: {STORY} linked trajectory and text",
)))


In [ ]:
# this cell shows the same story as a point inside the emotion triangle, scrubbable
bary_by_layer = {k: tp.barycentric(cos_by_layer[k]) for k in LAYERS}
trajectory_ternary_scrubber(
    LAYERS, bary_by_layer, emotions, starts, default_layer=33, tokens=TOKENS,
    title=f"{MODEL} | {STORY}<br><sub>expected: {PHASES} (softmax display transform, T=28)</sub>",
).show()


In [ ]:
# this cell renders the static three-layer small multiples for the print path
per_layer = [
    tp.smooth(tp.story_cosines(shard, labels, emotions, LAYERS.index(k), lineage="corpus"), window=8)
    for k in (6, 33, 51)
]
fig = tp.layer_ternaries(per_layer, [f"layer {k}" for k in (6, 33, 51)], emotions, tokens=TOKENS)
fig.update_layout(title=f"{MODEL} | {STORY} across layer bands<br><sub>expected: {PHASES}</sub>")
fig.show()


In [ ]:
# this cell plots the trajectory speed, the probe-free step-vs-ramp diagnostic
fig = tp.speed_figure(shard["speed"].astype(np.float32)[:, LAYERS.index(33)], starts, tokens=TOKENS)
fig.update_layout(title=f"{MODEL} | {STORY} | layer 33 | speed ||a(t) - a(t-1)||<br><sub>expected: {PHASES}</sub>")
fig.show()


**The confound check.** All 12 selfgen-lineage probes plus the triple's corpus probes over
the same story. The assigned emotion's row should dominate its own phase. Any off-triple
row that is hot everywhere would mean the probes read valence or style, not emotion
identity.


In [ ]:
# this cell builds the all-probe heatmap, the confound check, scrubbable across layers
BATTERY = ['happy', 'inspired', 'loving', 'proud', 'calm', 'desperate', 'angry', 'guilty', 'sad', 'afraid', 'nervous', 'surprised']  # scenario-test emotions, corpus lineage (no selfgen on base)
battery_idx = [labels.index(f"corpus:{e}") for e in BATTERY]
triple_idx = [labels.index(f"corpus:{e}") for e in emotions]
rows_idx = battery_idx + triple_idx
row_names = [labels[i] for i in rows_idx]
heat_by_layer = {}
for i, k in enumerate(LAYERS):
    d = shard["dots"].astype(np.float32)[:, i]
    d = d - d.mean(0, keepdims=True)
    nc = np.clip(shard["norms_centered"].astype(np.float32)[:, i][:, None], 1e-6, None)
    heat_by_layer[k] = d[:, rows_idx] / nc
trajectory_heatmap_scrubber(
    LAYERS, heat_by_layer, row_names, starts, default_layer=33, tokens=TOKENS,
    title=f"{MODEL} | {STORY} | all probes over tokens<br><sub>expected: {PHASES}</sub>",
).show()


**The animation.** The same trajectory as a playable sweep. Press Play, or drag the token
slider to hold the picture at any moment t; the path shown is everything up to t (every 2nd
token, layer 33). The faint line is the full path and diamonds are phase starts.


In [ ]:
# this cell plays the trajectory as an animation, with a token slider for moment-to-moment view
from emotion_vectors.interactive import trajectory_ternary_animation

layer = 33
phase_str = " -> ".join(f"{e} (t={s})" for e, s in zip(emotions, starts))
print(f"{MODEL} | {STORY} | layer {layer} | expected: {phase_str}")
trajectory_ternary_animation(
    bary_by_layer[layer], emotions, starts, tokens=TOKENS,
    title=f"{MODEL} | {STORY} | layer {layer}<br><sub>expected: {phase_str} (softmax display transform, T=28)</sub>",
).show()


**Layer and moment together (kernel tier).** The saved-notebook figures above give one
control each, because the frames mechanism behind a token slider cannot also swap layers.
With a running kernel the two compose: pick any layer band and any moment t, and see the
path traversed up to t at that layer, in both the triangle and the raw 3D view (the cone
arrow points in the direction of motion at t).


In [ ]:
# this cell composes both controls (layer x token) over the ternary AND the raw 3D view
import ipywidgets as widgets

def path_until(layer, t):
    cos = tp.smooth(
        tp.story_cosines(shard, labels, emotions, LAYERS.index(layer), lineage="corpus"), window=8
    )
    fig = tp.ternary_figure(cos[: t + 1], emotions, [s for s in starts if s <= t], tokens=TOKENS)
    fig.update_layout(
        title=f"{MODEL} | {STORY} | layer {layer} | path up to t={t}<br><sub>expected: {PHASES}</sub>"
    )
    fig.show()
    fig = tp.cosine_3d_figure(cos, emotions, [s for s in starts if s <= t], tokens=TOKENS, up_to=t)
    fig.update_layout(
        title=f"{MODEL} | {STORY} | layer {layer} | raw cosines, arrow at t={t}<br><sub>expected: {PHASES}</sub>"
    )
    fig.show()

widgets.interact(
    path_until,
    layer=widgets.SelectionSlider(options=LAYERS, value=33, description="layer"),
    t=widgets.IntSlider(min=5, max=len(shard["norms"]) - 1, value=len(shard["norms"]) - 1, step=5, description="t"),
);


**The bridge to Q1.** The same tokens projected on the circumplex plane: the first two
components of a principal component analysis (PCA) of the 171 corpus-lineage emotion means
at layer 33, the plane whose valence alignment H1 validated. The projection is computed
from the shard's stored probe dots alone (the PCA axes are re-expressed as weights over the
unit probes, an exact and tested identity), so it needs no activations.


In [ ]:
# this cell projects the story onto the Q1 circumplex plane, from shard dots alone
from emotion_vectors.trajectories import circumplex_weights

means_bundle = np.load(fetch("emotion_vectors/emotion_means.npz"), allow_pickle=True)
L33_FULL = list(means_bundle["layers"]).index(33)
weights = circumplex_weights(means_bundle["means"][:, L33_FULL].astype(np.float32))
corpus_idx = [i for i, name in enumerate(labels) if name.startswith("corpus:")]
d33 = shard["dots"].astype(np.float32)[:, LAYERS.index(33)][:, corpus_idx]
d33 = d33 - d33.mean(0, keepdims=True)
proj = tp.smooth((weights @ d33.T).T, window=8)
fig = tp.circumplex_figure(proj, starts, emotions, tokens=TOKENS)
fig.update_layout(title=f"{MODEL} | {STORY} | layer 33 | Q1 circumplex plane<br><sub>expected: {PHASES}</sub>")
fig.show()


**The untransformed view.** Raw centered cosines as 3D coordinates, no softmax, with a
layer slider. Color encodes token order. Read positions and traversal order, not distances:
the probe axes are oblique in activation space. The playable version below draws the path
token by token with a direction arrow; the composed kernel-tier widget above adds the layer
control.


In [ ]:
# this cell shows the raw 3D view, scrubbable across layers, with time direction
from emotion_vectors.interactive import trajectory_3d_scrubber

raw_by_layer = {
    k: tp.smooth(tp.story_cosines(shard, labels, emotions, i, lineage="corpus"), window=8)
    for i, k in enumerate(LAYERS)
}
trajectory_3d_scrubber(
    LAYERS, raw_by_layer, emotions, starts, default_layer=33, tokens=TOKENS,
    title=f"{MODEL} | {STORY} | raw cosines<br><sub>expected: {PHASES} (no display transform)</sub>",
).show()


In [ ]:
# this cell plays the 3D path drawing itself token by token, with a direction arrow
from emotion_vectors.interactive import trajectory_3d_animation

trajectory_3d_animation(
    raw_by_layer[33], emotions, starts, tokens=TOKENS,
    title=f"{MODEL} | {STORY} | layer 33 | raw cosines over time<br><sub>expected: {PHASES}</sub>",
).show()


## 3. One simultaneous story, for contrast

A SIMULTANEOUS story should show all three emotions co-active. In the ternary view that
means a trajectory hovering near the centroid rather than touring the corners.


In [ ]:
# this cell shows a SIMULTANEOUS story for contrast: it should hover near the centroid
sim = next(m for m in manifest if m["mode"] == "SIMULTANEOUS" and 150 < m["n_tokens"] < 260)
sshard = np.load(fetch(f"combined_trajectories_base/shards/{sim['story_id']}.npz"))
sim_phases = " + ".join(sim["emotions"]) + " (simultaneous, all three throughout)"
print(f"{MODEL} | {sim['story_id']} | expected: {sim_phases}")
sim_bary = {
    k: tp.barycentric(tp.smooth(tp.story_cosines(sshard, labels, sim["emotions"], i, lineage="corpus"), window=8))
    for i, k in enumerate(LAYERS)
}
trajectory_ternary_scrubber(
    LAYERS, sim_bary, sim["emotions"], [0], default_layer=33, tokens=decode_tokens(sshard),
    title=f"{MODEL} | {sim['story_id']}<br><sub>expected: {sim_phases}, hovering near the centroid</sub>",
).show()


In [ ]:
# this cell links the simultaneous story's text to its plots (single phase: the
# emotion-shading toggle is the interesting control here)
sim_cos_by_layer = {
    k: tp.smooth(tp.story_cosines(sshard, labels, sim["emotions"], i, lineage="corpus"), window=8)
    for i, k in enumerate(LAYERS)
}
display(HTML(linked_trajectory_html(
    sim_cos_by_layer, decode_tokens(sshard), sim["emotions"], [0], default_layer=33,
    show_ternary=True, title=f"{MODEL}: {sim['story_id']} linked trajectory and text",
)))


## 4. Explore more stories

Two exhibits are an anecdote, not a picture of the corpus. Two tiers, following the
explore_layers convention. The static tier works in the saved notebook; the kernel tier
covers everything.

**Static tier: a dropdown over 20 stratified stories** (2 per category x mode, drawn with a
fixed seed, layer 33). Sequential stories show the phase-crossover structure; simultaneous
ones show the co-active blur.


In [ ]:
# this cell samples 20 stratified stories and builds the dropdown browser
import random

from emotion_vectors.interactive import trajectory_story_dropdown

random.seed(20260722)
picks = []
for cat in sorted({m["category"] for m in manifest}):
    for mode in ("SEQUENTIAL", "SIMULTANEOUS"):
        pool = [
            m for m in manifest
            if m["category"] == cat and m["mode"] == mode and 150 < m["n_tokens"] < 300
            and (mode != "SEQUENTIAL" or len(m["phase_token_starts"]) == 3)
        ]
        picks += random.sample(pool, 2)

entries = []
for m in picks:
    z = np.load(fetch(f"combined_trajectories_base/shards/{m['story_id']}.npz"))
    cos = tp.smooth(
        tp.story_cosines(z, labels, m["phase_emotions"], LAYERS.index(33), lineage="corpus"),
        window=8,
    )
    entries.append({
        "label": f"{m['story_id']} | {m['mode'][:3]} | {' / '.join(m['phase_emotions'])}",
        "emotions": m["phase_emotions"],
        "cosines": cos,
        "starts": m["phase_token_starts"],
        "tokens": decode_tokens(z),
    })
trajectory_story_dropdown(
    entries, title=f"{MODEL}: 20 stories at layer 33"
).show()


**Kernel tier: every story, every layer, every moment.** The picker below spans the full
corpus (category, mode, story) and composes the other two controls: a layer slider over the
six captured bands and a token slider that holds the trajectory at moment t. The header
over the figures states the story, the layer, and the expected emotion order with its
transition tokens. The ternary shows the path traversed up to t; the lines view shows the
full curves with a cursor at t.


In [ ]:
# this cell defines explore(): any of the 5,888 stories as the linked view plus raw 3D
def explore(story_id, lineage="corpus"):
    """Linked plot+text view (all layers) and the raw 3D path for any story."""
    m = next((r for r in manifest if r["story_id"] == story_id), None)
    if m is None:
        print(f"story_id {story_id!r} not in the manifest")
        return
    z = np.load(fetch(f"combined_trajectories_base/shards/{story_id}.npz"))
    toks = decode_tokens(z)
    cbl = {
        k: tp.smooth(tp.story_cosines(z, labels, m["phase_emotions"], i, lineage=lineage), window=8)
        for i, k in enumerate(LAYERS)
    }
    phase_str = " -> ".join(f"{e} (t={s})" for e, s in zip(m["phase_emotions"], m["phase_token_starts"]))
    print(f"{story_id} | {MODEL} | {m['mode']} | expected: {phase_str}")
    display(HTML(linked_trajectory_html(
        cbl, toks, m["phase_emotions"], m["phase_token_starts"], default_layer=33,
        show_ternary=True, title=f"{MODEL}: {story_id} linked trajectory and text",
    )))
    fig = tp.cosine_3d_figure(cbl[33], m["phase_emotions"], m["phase_token_starts"], tokens=toks)
    fig.update_layout(title=f"{MODEL} | {story_id} | layer 33 | raw cosines")
    fig.show()

# example: the first sequential story of the 'D_timescale' category
example = next(m for m in manifest if m["category"] == "D_timescale" and m["mode"] == "SEQUENTIAL")
explore(example["story_id"])


In [ ]:
# this cell builds the full explorer: story picker; the linked view carries its own
# layer slider, and the t slider drives the 3D direction arrow
import ipywidgets as widgets

category_dd = widgets.Dropdown(options=sorted({m["category"] for m in manifest}), description="category")
mode_dd = widgets.Dropdown(options=["SEQUENTIAL", "SIMULTANEOUS"], description="mode")
story_dd = widgets.Dropdown(description="story")
t_sl = widgets.IntSlider(min=5, max=200, step=5, description="t")
_shard_cache = {}

def refresh_stories(*_):
    pool = [m for m in manifest if m["category"] == category_dd.value and m["mode"] == mode_dd.value]
    story_dd.options = [
        (f"{m['story_id']} | {' / '.join(m['phase_emotions'])}", m["story_id"]) for m in pool
    ]

def refresh_t_range(*_):
    m = next((r for r in manifest if r["story_id"] == story_dd.value), None)
    if m is not None:
        t_sl.max = m["n_tokens"] - 1
        t_sl.value = t_sl.max

def show_selected(story, t):
    # the story dropdown is transiently None while its options repopulate
    if story is None:
        print("pick a story")
        return
    m = next(r for r in manifest if r["story_id"] == story)
    if story not in _shard_cache:
        _shard_cache[story] = np.load(fetch(f"combined_trajectories_base/shards/{story}.npz"))
    z = _shard_cache[story]
    toks = decode_tokens(z)
    t = min(t, m["n_tokens"] - 1)
    cbl = {
        k: tp.smooth(tp.story_cosines(z, labels, m["phase_emotions"], i, lineage="corpus"), window=8)
        for i, k in enumerate(LAYERS)
    }
    phase_str = " -> ".join(f"{e} (t={s})" for e, s in zip(m["phase_emotions"], m["phase_token_starts"]))
    print(f"{MODEL} | {story} | expected: {phase_str}")
    display(HTML(linked_trajectory_html(
        cbl, toks, m["phase_emotions"], m["phase_token_starts"], default_layer=33,
        show_ternary=True, title=f"{MODEL}: {story} linked trajectory and text",
    )))
    fig = tp.cosine_3d_figure(cbl[33], m["phase_emotions"], [s for s in m["phase_token_starts"] if s <= t], tokens=toks, up_to=t)
    fig.update_layout(title=f"{MODEL} | {story} | layer 33 | raw cosines, arrow at t={t}")
    fig.show()

category_dd.observe(refresh_stories, "value")
mode_dd.observe(refresh_stories, "value")
story_dd.observe(refresh_t_range, "value")
refresh_stories()
refresh_t_range()
out = widgets.interactive_output(show_selected, {"story": story_dd, "t": t_sl})
display(widgets.HBox([category_dd, mode_dd, story_dd]), widgets.HBox([t_sl]), out)


## 5. Registered analysis (pending)

The population verdict comes from the transition-locked average: all sequential transitions
aligned at t=0, incoming vs outgoing curves, bootstrap bands, per layer band, scored
against the random-direction null. `transition_windows` and `transition_locked_figure` are
built and tested, but the reads (ramp-vs-step statistic, anticipation window, pass bars)
must be registered in TREE.md **before** that figure is rendered over the corpus. This
section will hold the registered analysis and its verdict; until then it is deliberately
empty.
